# Nutrient Analysis - Correlation with PCs

**Objective**: Analyze the relationship between PCA scores (representing dominant modes of environmental variability) and nutrient concentrations (DIN, DIP, TN, TP) across space.

## Nutrients of Interest
- **NH4-N** (Ammonium Nitrogen)
- **NO2-N** (Nitrite Nitrogen)
- **NO3-N** (Nitrate Nitrogen)
- **PO4-P** (Phosphate Phosphorus)
- **TN** (Total Nitrogen)
- **TP** (Total Phosphorus)
- **DIN** (Dissolved Inorganic Nitrogen = NH4 + NO2 + NO3)
- **DIP** (Dissolved Inorganic Phosphorus = PO4-P)

This notebook closely follows the logic of `12_PCA_analysis_continue.ipynb`.

In [1]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
from scipy.stats import pearsonr
import sys
from pathlib import Path
import shapely.wkb

# Add scripts to path
sys.path.append(str(Path.cwd().parent / 'scripts'))
from utils.paths import ProjectPaths

# Configure plots
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 300

In [2]:
# 1. Load Data
# Load PCA Scores
pca_scores_path = ProjectPaths.PROCESSED_DATA_DIR / 'pca_scores.csv'
pca_scores = pd.read_csv(pca_scores_path, index_col=0)

# Load Environmental Data (Cleaned)
env_data_path = ProjectPaths.PROCESSED_DATA_DIR / 'merged_gdf_saisuipoin_cleaned.feather'
env_data_df = pd.read_feather(env_data_path)

# Convert WKB to Geometry if necessary
try:
    if 'geometry' in env_data_df.columns:
        if isinstance(env_data_df['geometry'].iloc[0], bytes):
             env_data_df['geometry'] = env_data_df['geometry'].apply(lambda x: shapely.wkb.loads(x))
    
    env_data = gpd.GeoDataFrame(env_data_df, geometry='geometry', crs="EPSG:4326")
    print("Loaded Env Data with Geometry.")
except Exception as e:
    print(f"Error converting geometry: {e}")
    # Fallback if WKB fails but Lat/Lon exists
    if 'latitude' in env_data_df.columns and 'longitude' in env_data_df.columns:
        env_data = gpd.GeoDataFrame(
            env_data_df, 
            geometry=gpd.points_from_xy(env_data_df.longitude, env_data_df.latitude),
            crs="EPSG:4326"
        )
        print("Loaded Env Data with Lat/Lon fallback.")

# Ensure joining keys are consistent
if 'zettaicode' in env_data.columns:
    env_data['zettaicode'] = env_data['zettaicode'].astype(str)
pca_scores.index = pca_scores.index.astype(str)

# Filter for Summer (June, July, August)
env_data['month'] = env_data['surveymont']
summer_data = env_data[env_data['month'].isin([6, 7, 8])].copy()

print("Data Loaded:")
print(f"PCA Scores: {pca_scores.shape}")
print(f"Environmental Data (Summer): {summer_data.shape}")

Loaded Env Data with Geometry.
Data Loaded:
PCA Scores: (43, 43)
Environmental Data (Summer): (10694, 68)


In [3]:
# 2. Prepare Nutrient Variables per Station

# Define nutrients of interest
nutrients = ['nh4n', 'no2n', 'no3n', 'po4p', 'tn', 'tp']

# Ensure numeric types for aggregation
for col in nutrients:
    if col in summer_data.columns:
        summer_data[col] = pd.to_numeric(summer_data[col], errors='coerce')

# Calculate DIN and DIP
summer_data['din'] = summer_data['nh4n'].fillna(0) + summer_data['no2n'].fillna(0) + summer_data['no3n'].fillna(0)
summer_data['dip'] = summer_data['po4p']  # Assuming PO4-P represents DIP

# Update nutrient list
analysis_targets = nutrients + ['din', 'dip']

# Separate Surface and Bottom layers
surface_data = summer_data[summer_data['saisuipoin'] == 1]
bottom_data = summer_data[summer_data['saisuipoin'] == 2]

def get_station_time_series(df, var_col):
    # Get annual mean per station
    if 'surveyyear' in df.columns:
        df['year'] = df['surveyyear']
    elif 'date' in df.columns:
        df['year'] = pd.to_datetime(df['date']).dt.year
        
    matrix = df.groupby(['year', 'zettaicode'])[var_col].mean().unstack()
    return matrix

# Example: Extract matrix for DIN (Surface)
# We will iterate dynamically later, but checking one here
din_surface_matrix = get_station_time_series(surface_data, 'din')
print(f"DIN Surface Matrix Shape: {din_surface_matrix.shape}")

DIN Surface Matrix Shape: (43, 127)


/home/reyha/miniconda3/envs/sis_analysis/lib/python3.10/site-packages/geopandas/geodataframe.py:1969: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [4]:
# 3. Calculate Correlations

def calculate_correlations(pc_scores, env_matrix, pc_col='PC1'):
    correlations = {}
    p_values = {}

    pc_scores.index = pc_scores.index.astype(int)
    env_matrix.index = env_matrix.index.astype(int)
    
    common_years = pc_scores.index.intersection(env_matrix.index)
    
    for station in env_matrix.columns:
        s_env = env_matrix.loc[common_years, station]
        s_pc = pc_scores.loc[common_years]
        
        valid = s_env.notna() & s_pc.notna()
        
        if valid.sum() > 5:
            corr, p = pearsonr(s_pc[valid], s_env[valid])
            correlations[station] = corr
            p_values[station] = p
        else:
            correlations[station] = np.nan
            p_values[station] = np.nan
            
    return pd.Series(correlations, name=f'{pc_col}_Corr'), pd.Series(p_values, name=f'{pc_col}_Pval')

# Initialize DataFrame for results
# We'll use the station index from one of the matrices
station_corrs = pd.DataFrame(index=din_surface_matrix.columns)
station_corrs.index.name = 'zettaicode'

# Loop through Nutrients and Layers
layers = {'Surface': surface_data, 'Bottom': bottom_data}
pcs = ['PC1', 'PC2']

for layer_name, layer_df in layers.items():
    for nutrient in analysis_targets:
        # Get time series matrix
        matrix = get_station_time_series(layer_df, nutrient)
        
        for pc in pcs:
            if pc in pca_scores.columns:
                col_name = f'{pc}_{nutrient.upper()}_{layer_name}_Corr'
                r, p = calculate_correlations(pca_scores[pc], matrix, pc)
                station_corrs[col_name] = r

# Merge with Geometry for mapping
gdf_stations = env_data.drop_duplicates(subset='zettaicode')[['zettaicode', 'geometry']].set_index('zettaicode')
corr_map_data = gdf_stations.join(station_corrs, how='inner')

print("Correlation Matrix Head:")
print(corr_map_data.head())

/home/reyha/miniconda3/envs/sis_analysis/lib/python3.10/site-packages/geopandas/geodataframe.py:1969: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/home/reyha/miniconda3/envs/sis_analysis/lib/python3.10/site-packages/geopandas/geodataframe.py:1969: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/home/reyha/miniconda3/envs/sis_analysis/lib/python3.10/site-packages/geopandas/geodataframe.py:1969: SettingWithCopyWarning: 
A value is trying to be s

Correlation Matrix Head:
                              geometry  PC1_NH4N_Surface_Corr  \
zettaicode                                                      
00002811      POINT (135.33056 34.695)              -0.061809   
00002823    POINT (135.26389 34.65833)              -0.151330   
00000272    POINT (135.40917 34.65833)              -0.305693   
00002725    POINT (135.26778 34.54861)              -0.163656   
00000275    POINT (135.41222 34.55639)              -0.423437   

            PC2_NH4N_Surface_Corr  PC1_NO2N_Surface_Corr  \
zettaicode                                                 
00002811                 0.292530              -0.466735   
00002823                 0.195081              -0.296655   
00000272                 0.158471              -0.492417   
00002725                -0.286734              -0.637727   
00000275                 0.092825              -0.557086   

            PC2_NO2N_Surface_Corr  PC1_NO3N_Surface_Corr  \
zettaicode                            

In [5]:
# 4. Plot Correlation Maps

def plot_correlation_map(gdf, column, title, filename):
    fig, ax = plt.subplots(1, 1, figsize=(10, 8), subplot_kw={'projection': ccrs.PlateCarree()})
    
    # Add map features
    ax.add_feature(cfeature.LAND, facecolor='lightgray')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    
    # Plot data
    # Filter out NaNs for plotting
    plot_data = gdf.dropna(subset=[column])
    
    if not plot_data.empty:
        plot_data.plot(column=column, ax=ax, cmap='coolwarm', vmin=-1, vmax=1, 
                      legend=True, markersize=30, legend_kwds={'label': "Pearson Correlation"})
        ax.set_title(title)
        
        # Set extent (zoom to data)
        bounds = plot_data.total_bounds
        ax.set_extent([bounds[0]-0.5, bounds[2]+0.5, bounds[1]-0.5, bounds[3]+0.5])
        
        output_path = ProjectPaths.FIGURES_DIR / filename
        plt.savefig(output_path, bbox_inches='tight')
        print(f"Saved plot to {output_path}")
    else:
        print(f"No valid data for plotting {column}")
        
    plt.close()

# Plot for key metrics (e.g., DIN surface, DIP surface)
# We can loop, but let's pick a few illustrative ones to avoid too many files initially, or loop all.
# Let's generate maps for DIN and DIP (Surface and Bottom) against PC1 and PC2.

plot_vars = ['DIN', 'DIP', 'TN', 'TP']

for var in plot_vars:
    for layer in ['Surface', 'Bottom']:
        for pc in ['PC1', 'PC2']:
            col = f'{pc}_{var}_{layer}_Corr'
            if col in corr_map_data.columns:
                plot_correlation_map(corr_map_data, col, 
                                     f'{pc} vs {var} ({layer}) Correlation', 
                                     f'pca_corr_{pc}_{var}_{layer}.png')

Saved plot to /home/reyha/SIS_Analysis_WSL/figures/pca_corr_PC1_DIN_Surface.png
Saved plot to /home/reyha/SIS_Analysis_WSL/figures/pca_corr_PC2_DIN_Surface.png
Saved plot to /home/reyha/SIS_Analysis_WSL/figures/pca_corr_PC1_DIN_Bottom.png
Saved plot to /home/reyha/SIS_Analysis_WSL/figures/pca_corr_PC2_DIN_Bottom.png
Saved plot to /home/reyha/SIS_Analysis_WSL/figures/pca_corr_PC1_DIP_Surface.png
Saved plot to /home/reyha/SIS_Analysis_WSL/figures/pca_corr_PC2_DIP_Surface.png
Saved plot to /home/reyha/SIS_Analysis_WSL/figures/pca_corr_PC1_DIP_Bottom.png
Saved plot to /home/reyha/SIS_Analysis_WSL/figures/pca_corr_PC2_DIP_Bottom.png
Saved plot to /home/reyha/SIS_Analysis_WSL/figures/pca_corr_PC1_TN_Surface.png
Saved plot to /home/reyha/SIS_Analysis_WSL/figures/pca_corr_PC2_TN_Surface.png
Saved plot to /home/reyha/SIS_Analysis_WSL/figures/pca_corr_PC1_TN_Bottom.png
Saved plot to /home/reyha/SIS_Analysis_WSL/figures/pca_corr_PC2_TN_Bottom.png
Saved plot to /home/reyha/SIS_Analysis_WSL/figures